# Gold Layer - Dimensional Model for Retail Sales

## Purpose
This notebook creates a **star schema dimensional model** from the silver layer data for optimized analytics and reporting.

## Process Flow
1. **Read**: Load clean data from `sales.silver.retail_store_sales`
2. **Extract**: Create dimension tables with surrogate keys
3. **Transform**: Join dimensions to create fact table with foreign keys
4. **Write**: Save fact and dimension tables to gold layer

## Input
- Delta table: `sales.silver.retail_store_sales`

## Output

### Fact Table
- **sales.gold.sales_fact** - Transaction-level facts with foreign keys to dimensions
  - Measures: price_per_unit, quantity, total_spent
  - Dimensions: category_id, payment_id
  - Degenerate dimensions: transaction_id, customer_id, item_id, transaction_date

### Dimension Tables
- **sales.gold.category_dim** - Product categories
- **sales.gold.payment_dim** - Payment methods and locations

In [0]:
# ============================================================================
# Setup: Import Libraries
# ============================================================================

from pyspark.sql.functions import col, concat, lit, monotonically_increasing_id

In [0]:
# ============================================================================
# Step 1: Read Silver Layer Data
# ============================================================================
# Load clean, validated data from sales.silver.retail_store_sales

sales_silver_df = spark.read.table("sales.silver.retail_store_sales")
display(sales_silver_df)

In [0]:
# ============================================================================
# Step 2: Create Dimension Tables
# ============================================================================

# Category Dimension Table
# Extract unique categories and assign surrogate keys (CAT_0, CAT_1, ...)
category_dim_df = sales_silver_df \
                    .select("category") \
                    .distinct() \
                    .withColumn(
                        "category_id", 
                        concat(lit("CAT_"), monotonically_increasing_id().cast("string"))
                    ) \
                    .withColumnRenamed("category", "category_name")

display(category_dim_df)

# Payment Dimension Table
# Extract unique payment method and location combinations with surrogate keys
payment_dim_df = sales_silver_df \
                    .select("transaction_id", "payment_method", "location") \
                    .withColumn(
                        "payment_id", 
                        concat(lit("PYM_"), monotonically_increasing_id().cast("string"))
                    )

print(f"✓ Created payment dimension with {payment_dim_df.count()} records")
display(payment_dim_df)

In [0]:
# ============================================================================
# Step 3: Create Fact Table
# ============================================================================
# Join dimension tables to replace natural keys with surrogate keys
# Result: Fact table with measures and foreign keys to dimensions

sales_fact_df = sales_silver_df.alias("sales") \
                    .join(
                        category_dim_df.alias("cat"), 
                        col("sales.category") == col("cat.category_name"), 
                        "left") \
                    .join(
                        payment_dim_df.alias("pmt"), 
                        col("sales.transaction_id") == col("pmt.transaction_id"), 
                        "left") \
                    .drop("category", "payment_method", "location", "category_name", col("pmt.transaction_id"))

print(f"✓ Created sales fact table with {sales_fact_df.count()} transactions")
display(sales_fact_df)

In [0]:
# ============================================================================
# Step 4: Write to Gold Layer
# ============================================================================
# Save dimensional model tables to the gold layer

# Write Category Dimension
category_dim_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales.gold.category_dim")

# Write Payment Dimension
payment_dim_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales.gold.payment_dim")

# Write Sales Fact Table
sales_fact_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales.gold.sales_fact")